# Exploring the Dataset: Building the Hosts Table

**Goal:** Understand how we go from a raw YAML configuration file to a relational database table.

This notebook walks through:
1. Loading `servers.yaml` (the raw config that defines all 22 hosts in the network)
2. Examining what fields each host has
3. Transforming the data into a Pandas DataFrame
4. Mapping it to our planned `hosts` database table schema

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.  

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, just change `DATASET_ROOT` below.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell


## 1. Load the Raw YAML File

The file `processing/config/servers.yaml` defines every host in the simulated enterprise network.  
This is the source for our **`hosts`** database table.

In [2]:
import yaml

servers_file = DATASET_ROOT / "processing" / "config" / "servers.yaml"

with open(servers_file) as f:
    servers = yaml.safe_load(f)

print(f"Loaded {len(servers)} hosts from: {servers_file.name}")
print("\nHost names (keys in the YAML):")
for i, name in enumerate(servers.keys(), 1):
    print(f"  {i:2d}. {name}")

Loaded 22 hosts from: servers.yaml

Host names (keys in the YAML):
   1. remote_employee_1
   2. attacker_0
   3. remote_employee_2
   4. remote_employee_0
   5. cloud_share
   6. webserver
   7. morris_mail
   8. davey_mail
   9. vpn
  10. intranet_server
  11. internal_employee_1
  12. internal_employee_3
  13. ext_user_1
  14. ext_user_0
  15. ext_user_2
  16. internal_share
  17. monitoring
  18. internal_employee_2
  19. mail
  20. internal_employee_0
  21. inet-dns
  22. inet-firewall


## 2. Examine a Single Host

Let's look at what data exists for one host. We'll pick `intranet_server` because it's the main attack target in our dataset.

In [3]:
import json

example_host = "intranet_server"
host_data = servers[example_host]

print(f"Raw YAML data for '{example_host}':\n")
print(json.dumps(host_data, indent=2))

Raw YAML data for 'intranet_server':

{
  "hostname": "intranet-server",
  "groups": [
    "beatservers",
    "intranet",
    "servers"
  ],
  "distribution": "Ubuntu",
  "distribution_release": "bionic",
  "distribution_version": "18.04",
  "default_ipv4_address": "10.143.2.4",
  "default_ipv6_address": "fe80::f816:3eff:fe75:ba2b",
  "ipv4_addresses": [
    "10.143.2.4"
  ],
  "ipv6_addresses": [
    "fe80::f816:3eff:fe75:ba2b"
  ],
  "fqdns": [
    "intranet.smith.russellmitchell.com"
  ],
  "logs": [
    {
      "path": "apache2/*access*.log*",
      "type": "apache_access"
    },
    {
      "path": "apache2/*error*.log*",
      "type": "apache_error",
      "add_field": {
        "[@metadata][kyoushi][httpd_dirs]": [
          "/var/www/intranet.smith.russellmitchell.com",
          "",
          "/usr/share/javascript",
          "/javascript"
        ]
      }
    },
    {
      "path": "audit/audit.log*",
      "type": "audit",
      "add_field": {
        "[@metadata][pipeline

### What do these fields mean?

| YAML Field | What It Is | Example |
|-----------|-----------|--------|
| `hostname` | The machine's network name | `intranet-server` |
| `groups` | Roles/network zones this host belongs to | `[beatservers, intranet, servers]` |
| `username` | User account (only on employee workstations) | `mmorgan` |
| `distribution` | Linux distribution | `Ubuntu` |
| `distribution_version` | OS version | `18.04` |
| `default_ipv4_address` | Primary IPv4 address | `10.143.2.4` |
| `fqdns` | Fully qualified domain names | `[intranet.smith.russellmitchell.com]` |
| `logs` | What log files this host generates | List of log configs (path, type, etc.) |
| `timezone` | System timezone | `UTC` |

Not all hosts have the same fields. Employee workstations have `username` and `openvpn_user`; servers don't.

## 3. Transform into a DataFrame

Now we'll convert all 22 hosts into a structured table (Pandas DataFrame).  
This is exactly what we'll store in our PostgreSQL `hosts` table.

In [4]:
import pandas as pd

rows = []
for host_key, info in servers.items():
    rows.append(
        {
            "host_key": host_key,
            "hostname": info.get("hostname"),
            "groups": ", ".join(info.get("groups", [])),
            "username": info.get("username"),
            "os": info.get("distribution"),
            "os_version": info.get("distribution_version"),
            "default_ipv4": info.get("default_ipv4_address"),
            "default_ipv6": info.get("default_ipv6_address"),
            "fqdns": ", ".join(info.get("fqdns", [])),
            "log_types": ", ".join(
                sorted({log_cfg.get("type", "") for log_cfg in info.get("logs", [])})
            ),
            "num_log_sources": len(info.get("logs", [])),
        }
    )

df_hosts = pd.DataFrame(rows)
print(f"DataFrame shape: {df_hosts.shape[0]} rows x {df_hosts.shape[1]} columns")
print(f"\nColumns: {list(df_hosts.columns)}")

DataFrame shape: 22 rows x 11 columns

Columns: ['host_key', 'hostname', 'groups', 'username', 'os', 'os_version', 'default_ipv4', 'default_ipv6', 'fqdns', 'log_types', 'num_log_sources']


In [5]:
# Display the full table
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", None)
df_hosts

,host_key,hostname,groups,username,os,os_version,default_ipv4,default_ipv6,fqdns,log_types,num_log_sources
0,remote_employee_1,remote-employee-1,"employee, internet, remote_employee",twhite,Ubuntu,18.04,192.168.230.95,fe80::f816:3eff:fefb:8b25,,kyoushi,1
1,attacker_0,attacker-0,"attacker, internet",None,Ubuntu,18.04,192.168.230.122,fe80::f816:3eff:fe1d:2fc5,,"dnsteal, kyoushi, pcap",4
2,remote_employee_2,remote-employee-2,"employee, internet, remote_employee",jhall,Ubuntu,18.04,192.168.230.165,fe80::f816:3eff:fec4:1e0b,,kyoushi,1
3,remote_employee_0,remote-employee-0,"employee, internet, remote_employee",ahayes,Ubuntu,18.04,192.168.231.127,fe80::f816:3eff:feea:4536,,kyoushi,1
4,cloud_share,cloud-share,"dmz, proxied, servers",None,Ubuntu,18.04,172.19.130.106,fe80::f816:3eff:fe5b:275b,cloud.dmz.smith.russellmitchell.com,"apache_access, apache_error, audit, auth, syslog",5
5,webserver,webserver,"dmz, dnat, servers",None,Ubuntu,18.04,172.19.130.68,fe80::f816:3eff:fe47:ba82,proxy.smith.russellmitchell.com,"apache_access, apache_error, audit, auth, syslog",5
6,morris_mail,morris-mail,"ext_mail, internet, mailserver",None,Debian,9.11,192.168.231.164,fe80::f816:3eff:feb0:71cb,mailserver.morris.russellmitchell.com,"apache_access, apache_error, syslog",6
7,davey_mail,davey-mail,"ext_mail, internet, mailserver",None,Debian,9.11,192.168.231.56,fe80::f816:3eff:fe30:e67c,smtp.davey.russellmitchell.com,"apache_access, apache_error, syslog",6
8,vpn,vpn,"dmz, dnat, servers",None,Ubuntu,18.04,172.19.131.174,fe80::f816:3eff:fe1b:6d09,"vpn.smith.russellmitchell.com, vpn.dmz.smith.russellmitc...","audit, auth, openvpn, syslog",4
9,intranet_server,intranet-server,"beatservers, intranet, servers",None,Ubuntu,18.04,10.143.2.4,fe80::f816:3eff:fe75:ba2b,intranet.smith.russellmitchell.com,"apache_access, apache_error, audit, auth, syslog",5


In [6]:
# Query the DataFrame with SQL (pandasql runs SQL on the in-memory table)

from pandasql import sqldf

sqldf("SELECT hostname, default_ipv4, groups FROM df_hosts LIMIT 5")

,hostname,default_ipv4,groups
0,remote-employee-1,192.168.230.95,"employee, internet, remote_employee"
1,attacker-0,192.168.230.122,"attacker, internet"
2,remote-employee-2,192.168.230.165,"employee, internet, remote_employee"
3,remote-employee-0,192.168.231.127,"employee, internet, remote_employee"
4,cloud-share,172.19.130.106,"dmz, proxied, servers"


## 4. Understand the Network Zones

The hosts belong to different network zones. This is important for understanding how the attack moves through the network.

In [ ]:
# Count hosts by their group memberships
all_groups = []
for host_key, info in servers.items():
    for group in info.get("groups", []):
        all_groups.append({"host_key": host_key, "group": group})

df_groups = pd.DataFrame(all_groups)
print("Hosts per network group:\n")
print(df_groups.groupby("group")["host_key"].apply(list).to_string())

## 5. What Log Types Does Each Host Generate?

This helps us understand which raw log files we'll be parsing and loading into the database.

In [ ]:
# Show log types per host
print(f"{'Host':<25} {'# Logs':>7}   Log Types")
print("-" * 80)
for _, row in df_hosts.sort_values("num_log_sources", ascending=False).iterrows():
    print(f"{row['hostname']:<25} {row['num_log_sources']:>7}   {row['log_types']}")

## 6. Mapping to the Database Schema

Here's how this YAML data maps to our planned **`hosts`** table in PostgreSQL:

| YAML Field | DB Column | SQL Type | Notes |
|-----------|-----------|----------|-------|
| *(auto-generated)* | `host_id` | `SERIAL PRIMARY KEY` | Auto-incrementing ID |
| `hostname` | `hostname` | `VARCHAR(255) NOT NULL UNIQUE` | Machine's network name |
| `default_ipv4_address` | `ip_address` | `VARCHAR(45)` | Primary IPv4 (supports IPv6 length) |
| `groups` (derived) | `host_type` | `VARCHAR(50) NOT NULL` | Derived: firewall, server, workstation, attacker, etc. |
| `distribution` | `os_type` | `VARCHAR(100)` | Linux distribution name |
| `groups` (derived) | `network_zone` | `VARCHAR(50)` | Derived: dmz, intranet, internet |
| *(auto-generated)* | `created_at` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | When this row was inserted |

### The SQL `CREATE TABLE` statement:

```sql
CREATE TABLE hosts (
    host_id SERIAL PRIMARY KEY,
    hostname VARCHAR(255) NOT NULL UNIQUE,
    ip_address VARCHAR(45),
    host_type VARCHAR(50) NOT NULL,
    os_type VARCHAR(100),
    network_zone VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

Notice how `host_type` and `network_zone` don't exist directly in the YAML — we **derive** them from the `groups` list. That's a design decision we made during schema design.

In [ ]:
def derive_host_type(groups: list[str]) -> str:
    """Derive a host type from the groups list."""
    if "attacker" in groups:
        return "attacker"
    if "firewall" in groups:
        return "firewall"
    if "dnsservers" in groups:
        return "dns_server"
    if "mailserver" in groups:
        return "mail_server"
    if "servers" in groups or "beatservers" in groups:
        return "server"
    if "employee" in groups:
        return "workstation"
    if "ext_user" in groups:
        return "external_user"
    return "unknown"


def derive_network_zone(groups: list[str]) -> str:
    """Derive network zone from the groups list."""
    if "internet" in groups:
        return "external"
    if "dmz" in groups:
        return "dmz"
    if "intranet" in groups:
        return "internal"
    return "unknown"


# Build the table as it would look in the database
db_rows = []
for i, (_host_key, info) in enumerate(servers.items(), 1):
    groups = info.get("groups", [])
    db_rows.append(
        {
            "host_id": i,
            "hostname": info.get("hostname"),
            "ip_address": info.get("default_ipv4_address"),
            "host_type": derive_host_type(groups),
            "os_type": info.get("distribution"),
            "network_zone": derive_network_zone(groups),
        }
    )

df_hosts_table = pd.DataFrame(db_rows)
print("Preview of the 'hosts' table as it will look in PostgreSQL:\n")
df_hosts_table

## 7. Summary

What we just did:

1. **Loaded** a raw YAML config file (`servers.yaml`) that defines the network
2. **Examined** the structure — each host has hostname, IPs, groups, OS info, and log configs
3. **Transformed** it into a flat DataFrame with all 22 hosts
4. **Derived** new columns (`host_type`, `network_zone`) that don't exist in the raw data
5. **Mapped** it to the planned SQL schema for the `hosts` table

This is the process for **every table** in our database:
- Raw log file → Parse → DataFrame → Validate → Load into PostgreSQL

### Next Steps

- Explore the raw log files (Apache, DNS, VPN, etc.) using the same approach
- Understand how each log type maps to its own table (`http_events`, `dns_events`, etc.)
- Load the data into PostgreSQL and run SQL queries